# Bilinear system from the paper

This notebook runs `verify_stability` on the two-dimensional bilinear benchmark
of the paper and records how the certified rate changes during refinement. The
field is written with `jax.numpy`, so the compiled JAX kernel is used.

Requirements: Python 3.10 or later and
`pip install "pyddrv[jax,examples] @ git+https://github.com/NetDLab/pyDDRV"`.
See `examples/notebooks/README.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_stability
from pyddrv.systems.fields_jax import bilinear_2d_jax

## The system

$$\dot x = A x + B_1\,[x_1^2,\ x_1 x_2,\ x_2^2]^\top,\qquad
A = \begin{bmatrix} 0 & 2 \\ -1 & -1\end{bmatrix},$$

with the entries of `B_1` drawn independently from a normal distribution with
standard deviation `eta = 0.3`. The Jacobian is affine in the state. With the
analytic Jacobian supplied (`jac=`), the default `L_method="auto"` evaluates the
matrix measure at the corners of the box that contains the reachable set, where
its maximum lies, so the Lipschitz bound is exact.

In [ ]:
f, jac = bilinear_2d_jax(eta=0.3, seed=0)     # JAX field + analytic Jacobian
R, d, tau = 0.7, 2, 5.0

## Certification

`delta=0.05` asks for refinement until the certified rate is within 5% of
`alpha_upper`, with at most 60 seconds of computation. `record_trace=True`
stores the rate after each refinement round.

In [ ]:
import time
t0 = time.time()
report = verify_stability(f, R=R, d=d, jac=jac, tau=tau, eps=0.01,
                          delta=0.05, max_refine=16, max_seconds=60,
                          record_trace=True)
print(report.summary())
print(f"wall time {time.time()-t0:.1f}s, backend={report.backend}")

The certified rate is about 0.47. The paper reports a similar value for another
random draw of `B_1` with the same distribution; `seed=0` is not the paper's
draw.

## Certified rate during refinement

The first rounds use a coarse grid and certify a negative rate. As cubes are
split the rate rises toward the dashed line, `alpha_upper`. Every point on the
curve is a valid lower bound.

In [ ]:
from pyddrv.viz import plot_anytime

ax = plot_anytime(report, label="certified rate (anytime)")
ax.set_title("Bilinear 2D: anytime certified decay rate")
plt.show()

In [ ]:
# the raw trace, if you want the numbers
import numpy as np
np.array(report.trace)      # columns: wall-time [s], certified alpha, n_cubes

## The cubes

The same plots as in the pendulum notebook. With `delta=0.05` the refinement
goes deeper than it does there.

In [ ]:
from pyddrv.viz import plot_stability_2d

plot_stability_2d(report, f=f, show_grid=True, grid_color_by="outline")
plt.title("Covering grid over the flow"); plt.show()

plot_stability_2d(report, show_grid=True, grid_color_by="width")
plt.title("Cube widths (log scale)"); plt.show()